# 2026 COMP90042 Project — Group 073
*Fact-checking: two-stage retrieval (BM25 → BERT cross-encoder) + claim classifier.*

# Readme

**Before running:**

1. From your local machine, push the working branch to GitHub:
   `git push -u origin vita/google-colab`
2. Add a GitHub Personal Access Token to Colab Secrets:
   sidebar 🔑 → **Add new secret** → name `GITHUB_PAT`, value = your fine-grained PAT with **Contents: Read** on the repo, allow notebook access.
3. On Google Drive at `/content/drive/MyDrive/COMP90042_2026/`, place the data:
   - Required: `data/evidence.json` (~1 GB), `data/train-claims.json`, `data/dev-claims.json`, `data/test-claims-unlabelled.json`
   - Optional (saves ~30 min): `cache/bm25_index/`, `cache/bm25_train_top200.json`

Cell 1.1 clones the code from GitHub to `/content/COMP90042_2026` (Colab's fast local SSD) and symlinks `data/`, `cache/`, `checkpoints/`, `outputs/` from Drive — so code is git-managed and data persists across sessions.

**To resume training after a Colab disconnect:** set `RESUME_FROM` in cell 2.1 to the last saved epoch checkpoint, e.g. `'checkpoints/cross_encoder_epoch1.pt'`.

**Pipeline:** BM25 first-stage retrieval (1.2M → 200 candidates) → BERT cross-encoder reranker (top-4) → claim classifier.


# 1.DataSet Processing
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [4]:
# @title 1.1 · Setup — Sync code from GitHub, mount Drive, install packages

import os

# Stop JAX/TF from preallocating GPU memory. Colab pre-loads JAX which by
# default grabs ~75% of T4 VRAM, causing CUDA OOM when PyTorch then tries
# to allocate. These env vars MUST be set before any GPU library is imported.
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

import shutil
import subprocess
import sys
import json
import re
import random
from pathlib import Path
from google.colab import drive, userdata

drive.mount("/content/drive")

# -- EDIT IF NEEDED ----------------------------------------------------------
GITHUB_USER = "VitaChien"
REPO_NAME = "COMP90042_2026"
BRANCH = "vita/retriever"
DRIVE_DATA = "/content/drive/MyDrive/COMP90042_2026"  # data + checkpoints persist here
PROJECT_ROOT = "/content/COMP90042_2026"  # code clones to fast local SSD
# ----------------------------------------------------------------------------

# Pull latest code from GitHub (clone on first run, pull on subsequent runs)
pat = userdata.get("GITHUB_PAT")
repo_url = f"https://@github.com/{GITHUB_USER}/{REPO_NAME}.git"
if not os.path.exists(PROJECT_ROOT):
    subprocess.check_call(["git", "clone", "-b", BRANCH, repo_url, PROJECT_ROOT])
else:
    subprocess.check_call(["git", "-C", PROJECT_ROOT, "pull"])

# Symlink data directories from Drive into the cloned repo so data/checkpoints
# persist across Colab sessions while code stays on the fast local SSD.
for sub in ("data", "cache", "checkpoints", "outputs"):
    src = f"{DRIVE_DATA}/{sub}"
    dst = f"{PROJECT_ROOT}/{sub}"
    os.makedirs(src, exist_ok=True)
    if os.path.exists(dst) and not os.path.islink(dst):
        shutil.rmtree(dst)  # replace tracked .gitkeep dir with symlink
    if not os.path.islink(dst):
        os.symlink(src, dst)

# Install Python dependencies
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "torch",
        "transformers>=4.40",
        "bm25s[full]>=0.3",
        "scikit-learn>=1.3",
        "tqdm>=4.65",
    ]
)

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print(f"Working directory: {os.getcwd()}")



# Classifier lib import
from typing import Dict, List, Tuple, Callable
from collections import Counter

import numpy as np
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

# Transform labels to numeric type
LABEL2ID = {
    "SUPPORTS": 0,
    "REFUTES": 1,
    "NOT_ENOUGH_INFO": 2,
    "DISPUTED": 3
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

# Check device type
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/COMP90042_2026
Using device: cuda


In [5]:
# @title 1.2 · Verify data files exist

root = Path(PROJECT_ROOT)
checks = [
    ("data/evidence.json", "required (~1 GB)"),
    ("data/train-claims.json", "required"),
    ("data/dev-claims.json", "required"),
    ("data/test-claims-unlabelled.json", "required"),
    ("cache/bm25_index", "optional — saves ~10 min"),
    ("cache/bm25_train_top200.json", "optional — saves ~20 min"),
]
missing_required = False
for rel, note in checks:
    p = root / rel
    if p.exists():
        size = f"{p.stat().st_size/1e6:.0f} MB" if p.is_file() else "dir"
        print(f"  OK  {rel:45s} {size:10s}  ({note})")
    else:
        icon = "XX" if "required" in note else "--"
        print(f'  {icon}  {rel:45s} {"MISSING":10s}  ({note})')
        if "required" in note:
            missing_required = True

if missing_required:
    raise FileNotFoundError("Required data files are missing. Upload them to Drive first.")

  OK  data/evidence.json                            174 MB      (required (~1 GB))
  OK  data/train-claims.json                        0 MB        (required)
  OK  data/dev-claims.json                          0 MB        (required)
  OK  data/test-claims-unlabelled.json              0 MB        (required)
  --  cache/bm25_index                              MISSING     (optional — saves ~10 min)
  --  cache/bm25_train_top200.json                  MISSING     (optional — saves ~20 min)


In [6]:
# @title 1.3 · Set seed & utility function
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)


def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def normalise_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s\.\,\-\%°]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def simple_tokenise(text: str) -> List[str]:
    return normalise_text(text).split()


def concatenate_evidence(evidence_ids: List[str], evidence_corpus: Dict[str, str], max_evidence: int = 4) -> str:
    evidence_texts = []
    for eid in evidence_ids[:max_evidence]:
        if eid in evidence_corpus:
            evidence_texts.append(evidence_corpus[eid])

    if len(evidence_texts) == 0:
        return "No relevant evidence found."

    return " ".join(evidence_texts)

In [7]:
# @title 1.5 Read files and set paths
DATA_DIR = Path(PROJECT_ROOT) / "data"
OUTPUT_DIR = Path(PROJECT_ROOT) / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EVIDENCE_PATH = DATA_DIR / "evidence.json"
TRAIN_PATH = DATA_DIR / "train-claims.json"
DEV_PATH = DATA_DIR / "dev-claims.json"
TEST_PATH = DATA_DIR / "test-claims-unlabelled.json"

evidence_corpus = load_json(EVIDENCE_PATH)
train_claims = load_json(TRAIN_PATH)
dev_claims = load_json(DEV_PATH)

print("Evidence size:", len(evidence_corpus))
print("Train claims:", len(train_claims))
print("Dev claims:", len(dev_claims))

Evidence size: 1208827
Train claims: 1228
Dev claims: 154


In [8]:
# @title 1.4 · Build BM25 index — idempotent, skips if cache/bm25_index/ already exists
# Takes ~10 min on first run; subsequent runs print 'already exists' and return.
import importlib.util

spec = importlib.util.spec_from_file_location("build_bm25", f"{PROJECT_ROOT}/scripts/build_bm25.py")
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
mod.main()

[11:37:42] bm25-build INFO | Loading evidence ...
[11:37:44] bm25-build INFO | load_evidence took 2.34s
[11:37:44] bm25-build INFO | Loaded 1208827 evidence passages
[11:37:44] bm25 INFO | Tokenizing 1208827 evidences for BM25 ...


tokenize: 100%|██████████| 1208827/1208827 [00:38<00:00, 31229.45it/s]

[11:38:23] bm25 INFO | Building bm25s index ...



DEBUG:bm25s:Building index from tokens


[11:38:58] bm25 INFO | Saved BM25 index -> /content/COMP90042_2026/cache/bm25_index (152.2 MB across 6 files)
[11:38:59] bm25-build INFO | build_index took 74.69s


# 2.Model Implementation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [9]:
# @title 2.1 · Train cross-encoder (configure RESUME_FROM / EPOCHS, then run)
#
# After a Colab disconnect:
#   1. Re-run cell 1.1 (will git-pull latest code and re-mount Drive).
#   2. Set RESUME_FROM below to the last saved epoch checkpoint path.
#   3. Re-run this cell.

import importlib.util
import sys

RESUME_FROM = None  # e.g. "checkpoints/cross_encoder_epoch1.pt"  (None = start fresh)
EPOCHS = 4  # override cfg.ce_epochs=2: with hard_negatives_per_pos=8 (~37k pairs)
# the loss was still dropping at epoch 2 in earlier runs.

for key in list(sys.modules):
    if key.startswith(("src.", "scripts.")):
        del sys.modules[key]

spec = importlib.util.spec_from_file_location(
    "train_cross_encoder", f"{PROJECT_ROOT}/scripts/train_cross_encoder.py"
)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

sys.argv = ["train_cross_encoder.py"]
if RESUME_FROM:
    sys.argv += ["--resume", RESUME_FROM]
if EPOCHS is not None:
    sys.argv += ["--epochs", str(EPOCHS)]

mod.main()
print("\nDone. Final checkpoint: checkpoints/cross_encoder.pt")

[11:39:11] ce-train INFO | Training on device: cuda
[11:39:11] ce-train INFO | Loaded 1228 train claims
[11:39:18] ce-train INFO | BM25 search over train split took 5.94s
[11:39:19] ce-train INFO | Cached BM25 train top-200 -> /content/COMP90042_2026/cache/bm25_train_top200.json
[11:39:19] ce-train INFO | Built 37098 training pairs (positives=4122, hard_negs=32976)
[11:39:19] ce-train INFO | Loading evidence corpus into memory (~1.2M items, ~1GB) ...
[11:39:21] ce-train INFO | load_evidence took 2.15s
[11:39:21] ce-train INFO | Loaded 1208827 evidences


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

[12:21:30] cross-enc INFO | epoch 3 mean_loss=0.0977


[12:21:37] cross-enc INFO | Saved epoch checkpoint -> /content/COMP90042_2026/checkpoints/cross_encoder_epoch3.pt


CE epoch 4/4: 100%|██████████| 290/290 [13:52<00:00,  2.87s/it]

[12:35:30] cross-enc INFO | epoch 4 mean_loss=0.0678


[12:35:38] cross-enc INFO | Saved epoch checkpoint -> /content/COMP90042_2026/checkpoints/cross_encoder_epoch4.pt
[12:35:43] cross-enc INFO | Saved cross-encoder ckpt -> /content/COMP90042_2026/checkpoints/cross_encoder.pt

Done. Final checkpoint: checkpoints/cross_encoder.pt


In [10]:
# @title 2.2 · Run retriever-only inference on dev + test + train (BM25 top-200 -> CE rerank top-K)
# Loads BM25 index, cross-encoder, and the ~1GB evidence corpus ONCE,
# then loops over splits. Skips any split whose output file already exists,
# so a Colab disconnect mid-run can be resumed by simply re-running the cell.
# Order: dev -> test -> train (cheap first; train is ~1228 claims, slowest).
import importlib.util
import sys


SPLITS = ["dev", "test", "train"]
TOP_K = 4
BM25_TOP_K = 200

for key in list(sys.modules):
    if key.startswith(("src.", "scripts.")):
        del sys.modules[key]

spec = importlib.util.spec_from_file_location(
    "run_inference", f"{PROJECT_ROOT}/scripts/run_inference.py"
)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

cfg = mod.Config()
split_to_path = {"train": cfg.train_path, "dev": cfg.dev_path, "test": cfg.test_path}

# Decide which splits actually need running before paying the load cost.
pending = []
for split in SPLITS:
    out = cfg.output_dir / f"{split}-retriever-only-k{TOP_K}-bm25{BM25_TOP_K}.json"
    if out.exists():
        print(f"[{split}] skip - {out.name} already exists")
    else:
        pending.append((split, out))

if not pending:
    print("\nAll three split outputs already present. Delete a file under outputs/ to force a rerun.")
else:
    bm25, ce_tok, ce_model, evidence, device = mod.load_retriever_components(cfg)
    print(f"\nLoaded components on device={device}; running {len(pending)} split(s)...")
    for split, out in pending:
        print(f"\n=== {split} -> {out.name} ===")
        mod.run_retriever_only(
            split_to_path[split], out, TOP_K, bm25_top_k=BM25_TOP_K,
            bm25=bm25, ce_tok=ce_tok, ce_model=ce_model, evidence=evidence, device=device,
        )
        if split in {"train", "dev"}:
            m = mod.evaluate_predictions(out, split_to_path[split])
            print(f"[{split}] F={m['evidence_f']:.4f}  A={m['claim_accuracy']:.4f}  HM={m['harmonic_mean']:.4f}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[12:56:28] infer INFO | Loading evidence corpus ...
[12:56:30] infer INFO | load_evidence took 1.93s

Loaded components on device=cuda; running 3 split(s)...

=== dev -> dev-retriever-only-k4-bm25200.json ===
[12:56:30] infer INFO | Reranking on device: cuda
[12:56:30] infer INFO | BM25 pool size: 200


rerank: 100%|██████████| 154/154 [04:18<00:00,  1.68s/it]

[13:00:48] infer INFO | Retriever pipeline x154 took 258.19s
[13:00:48] infer INFO | Saved 154 predictions -> /content/COMP90042_2026/outputs/dev-retriever-only-k4-bm25200.json
[dev] F=0.2017  A=0.2468  HM=0.2219

=== test -> test-retriever-only-k4-bm25200.json ===
[13:00:48] infer INFO | Reranking on device: cuda
[13:00:48] infer INFO | BM25 pool size: 200



rerank: 100%|██████████| 153/153 [04:24<00:00,  1.73s/it]

[13:05:13] infer INFO | Retriever pipeline x153 took 264.58s
[13:05:13] infer INFO | Saved 153 predictions -> /content/COMP90042_2026/outputs/test-retriever-only-k4-bm25200.json

=== train -> train-retriever-only-k4-bm25200.json ===
[13:05:13] infer INFO | Reranking on device: cuda
[13:05:13] infer INFO | BM25 pool size: 200



rerank: 100%|██████████| 1228/1228 [35:20<00:00,  1.73s/it]

[13:40:33] infer INFO | Retriever pipeline x1228 took 2120.36s
[13:40:33] infer INFO | Saved 1228 predictions -> /content/COMP90042_2026/outputs/train-retriever-only-k4-bm25200.json
[train] F=0.3132  A=0.2606  HM=0.2845


In [11]:
# @title 2.3  Read retriever json files for classifier

# ---------------- Read retriever json files --------------------
# Fixed retriever output for train/dev claims.
TRAIN_RETRIEVER_JSON_PATH = OUTPUT_DIR / "train-retriever-only-k4-bm25200.json"
DEV_RETRIEVER_JSON_PATH = OUTPUT_DIR / "dev-retriever-only-k4-bm25200.json"

# Optional test retriever
TEST_RETRIEVER_JSON_PATH = OUTPUT_DIR / "test-retriever-only-k4-bm25200.json"

# ------------------retriever setup----------------
class FixedJSONRetriever:
    """Use a precomputed retriever-output JSON file as the retriever.

    Expected format:
    {
      "claim-752": {
        "claim_text": "...",
        "claim_label": "SUPPORTS",
        "evidences": ["evidence-1", "evidence-2", ...]
      }
    }
    """

    def __init__(self, retriever_json_path: Path):
        self.retriever_json_path = Path(retriever_json_path)
        self.retriever_results = load_json(self.retriever_json_path)

        # Fallback lookup for cases where only claim_text is passed.
        self.text_to_evidences = {}
        for claim_id, item in self.retriever_results.items():
            claim_text = item.get("claim_text", "")
            self.text_to_evidences[normalise_text(claim_text)] = item.get("evidences", [])

        print("Loaded JSON retriever:", self.retriever_json_path)
        print("Claims with retrieved evidence:", len(self.retriever_results))

    def retrieve_by_claim_id(self, claim_id: str, top_k: int = 4) -> List[str]:
        if claim_id not in self.retriever_results:
            return []
        return self.retriever_results[claim_id].get("evidences", [])[:top_k]



train_json_retriever = FixedJSONRetriever(TRAIN_RETRIEVER_JSON_PATH)
dev_json_retriever = FixedJSONRetriever(DEV_RETRIEVER_JSON_PATH)


Loaded JSON retriever: /content/COMP90042_2026/outputs/train-retriever-only-k4-bm25200.json
Claims with retrieved evidence: 1228
Loaded JSON retriever: /content/COMP90042_2026/outputs/dev-retriever-only-k4-bm25200.json
Claims with retrieved evidence: 154


In [15]:
# @title 2.4 build vocabulary & claim-evidence dataset

# ------------------ build vocabulary--------------------
def build_vocab(
    claims_json,
    evidence_corpus,
    retriever=None,
    min_freq=2,
    max_vocab_size=50000,
    retrieval_top_k=4,
    include_gold_evidence=True,
    include_retrieved_evidence=True
):
    """
    Build vocabulary from the training split only.

    Included text sources:
    1. claim text
    2. gold evidence IDs from train_claims, if available
    3. retrieved evidence IDs from the fixed JSON retriever

    This is useful because your classifier is trained with retrieved evidence,
    so the vocabulary should also see words from both retrieved evidence and gold evidence.
    """
    counter = Counter()

    for claim_id, instance in claims_json.items():
        # 1. Claim text
        counter.update(simple_tokenise(instance["claim_text"]))

        evidence_ids_for_vocab = []

        # 2. Gold evidence from training labels
        if include_gold_evidence:
            evidence_ids_for_vocab.extend(instance.get("evidences", []))

        # 3. Retrieved evidence from fixed JSON retriever
        if include_retrieved_evidence and retriever is not None:
            if hasattr(retriever, "retrieve_by_claim_id"):
                retrieved_ids = retriever.retrieve_by_claim_id(
                    claim_id,
                    top_k=retrieval_top_k
                )
            else:
                retrieved_ids = retriever.retrieve(
                    instance["claim_text"],
                    top_k=retrieval_top_k
                )

            evidence_ids_for_vocab.extend(retrieved_ids)

        # Remove duplicate evidence IDs for the same claim
        evidence_ids_for_vocab = list(dict.fromkeys(evidence_ids_for_vocab))

        for eid in evidence_ids_for_vocab:
            if eid in evidence_corpus:
                counter.update(simple_tokenise(evidence_corpus[eid]))

    vocab = {
        "<PAD>": 0,
        "<UNK>": 1,
        "<CLAIM>": 2,
        "<EVIDENCE>": 3
    }

    for word, freq in counter.most_common(max_vocab_size):
        if freq >= min_freq and word not in vocab:
            vocab[word] = len(vocab)

    return vocab


vocab = build_vocab(
    claims_json=train_claims,
    evidence_corpus=evidence_corpus,
    retriever=train_json_retriever,
    min_freq=2,
    max_vocab_size=50000,
    retrieval_top_k=4,
    include_gold_evidence=True,
    include_retrieved_evidence=True
)

print("Vocab size:", len(vocab))


# ----------------- claim + evidence embedding -----------------
def encode_tokens(tokens, vocab, max_len=512):
    ids = [vocab.get(tok, vocab["<UNK>"]) for tok in tokens]
    return ids[:max_len]


def encode_claim_evidence(claim_text, evidence_text, vocab, max_len=512):
    tokens = (
        ["<CLAIM>"]
        + simple_tokenise(claim_text)
        + ["<EVIDENCE>"]
        + simple_tokenise(evidence_text)
    )
    return encode_tokens(tokens, vocab, max_len=max_len)


# ------------------ claim-evidence dataset set up----------------
class ClaimEvidenceDataset(Dataset):
    def __init__(
        self,
        claims_json,
        evidence_corpus,
        vocab,
        max_len=384,
        max_evidence=4,
        evidence_mode="retrieved",
        use_gold_evidence=None,
        retriever=None,
        retrieval_top_k=4,
        is_test=False
    ):
        """
        evidence_mode controls which evidence text is used as model input.

        Options:
        - "gold": use only gold evidence IDs from the labelled claims file
        - "retrieved": use only evidence IDs from the fixed JSON retriever
        - "gold_and_retrieved": concatenate gold evidence IDs and retrieved evidence IDs

        For dev/test prediction, "retrieved" is usually the realistic setting (gold evidence not available for the unlabelled test set)
        """
        self.items = list(claims_json.items())
        self.evidence_corpus = evidence_corpus
        self.vocab = vocab
        self.max_len = max_len
        self.max_evidence = max_evidence

        # Backward compatibility with the previous boolean argument.
        if use_gold_evidence is not None:
            evidence_mode = "gold" if use_gold_evidence else "retrieved"

        valid_modes = {"gold", "retrieved", "gold_and_retrieved"}
        if evidence_mode not in valid_modes:
            raise ValueError(f"evidence_mode must be one of {valid_modes}, got {evidence_mode}")

        self.evidence_mode = evidence_mode
        self.retriever = retriever
        self.retrieval_top_k = retrieval_top_k
        self.is_test = is_test

    def __len__(self):
        return len(self.items)

    def _get_gold_evidence_ids(self, instance):
        if self.is_test:
            return []
        return instance.get("evidences", [])[:self.retrieval_top_k]

    def _get_retrieved_evidence_ids(self, claim_id, claim_text):
        if self.retriever is None:
            raise ValueError(
                "A retriever is required when evidence_mode uses retrieved evidence."
            )

        if hasattr(self.retriever, "retrieve_by_claim_id"):
            return self.retriever.retrieve_by_claim_id(
                claim_id,
                top_k=self.retrieval_top_k
            )

        return self.retriever.retrieve(
            claim_text,
            top_k=self.retrieval_top_k
        )

    def __getitem__(self, idx):
        claim_id, instance = self.items[idx]
        claim_text = instance["claim_text"]

        gold_ids = []
        retrieved_ids = []

        if self.evidence_mode in {"gold", "gold_and_retrieved"}:
            gold_ids = self._get_gold_evidence_ids(instance)

        if self.evidence_mode in {"retrieved", "gold_and_retrieved"}:
            retrieved_ids = self._get_retrieved_evidence_ids(claim_id, claim_text)

        # Put gold evidence first because it is usually more reliable,
        # then add retrieved evidence. Remove duplicates while preserving order.
        evidence_ids = list(dict.fromkeys(gold_ids + retrieved_ids))

        evidence_text = concatenate_evidence(
            evidence_ids=evidence_ids,
            evidence_corpus=self.evidence_corpus,
            max_evidence=self.max_evidence
        )

        input_ids = encode_claim_evidence(
            claim_text=claim_text,
            evidence_text=evidence_text,
            vocab=self.vocab,
            max_len=self.max_len
        )

        item = {
            "claim_id": claim_id,
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "evidence_ids": evidence_ids[:self.max_evidence]
        }

        if not self.is_test:
            item["label"] = torch.tensor(LABEL2ID[instance["claim_label"]], dtype=torch.long)

        return item


def collate_fn(batch):
    input_ids = [item["input_ids"] for item in batch]
    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=vocab["<PAD>"])
    attention_mask = (input_ids != vocab["<PAD>"]).long()

    output = {
        "claim_ids": [item["claim_id"] for item in batch],
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "evidence_ids": [item["evidence_ids"] for item in batch]
    }

    if "label" in batch[0]:
        output["labels"] = torch.stack([item["label"] for item in batch])

    return output


Vocab size: 8808


In [16]:
# @title 2.5 classifier design (CNN + BiLSTM + Multihead attention)

class AttentionPooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, hidden_states, attention_mask=None):
        scores = self.attention(hidden_states).squeeze(-1)

        if attention_mask is not None:
            scores = scores.masked_fill(~attention_mask.bool(), -1e9)

        weights = torch.softmax(scores, dim=1)
        pooled = torch.sum(hidden_states * weights.unsqueeze(-1), dim=1)
        return pooled


class CNNBiLSTMMultiheadClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        cnn_channels=64,
        kernel_sizes=(3, 5, 7),
        lstm_hidden_dim=128,
        lstm_layers=1,
        num_labels=4,
        num_heads=4,
        dropout=0.3,
        pad_idx=0
    ):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.embedding_dropout = nn.Dropout(dropout)

        self.convs = nn.ModuleList([
            nn.Conv1d(embedding_dim, cnn_channels, kernel_size=k, padding=k // 2)
            for k in kernel_sizes
        ])

        cnn_output_dim = cnn_channels * len(kernel_sizes)

        self.bilstm = nn.LSTM(
            input_size=cnn_output_dim,
            hidden_size=lstm_hidden_dim,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0.0
        )

        lstm_output_dim = lstm_hidden_dim * 2
        self.multihead_attn = nn.MultiheadAttention(
            embed_dim=lstm_output_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        self.layer_norm = nn.LayerNorm(lstm_output_dim)
        self.attention_pooling = AttentionPooling(lstm_output_dim)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(lstm_output_dim, lstm_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden_dim, num_labels)
        )

    def forward(self, input_ids, attention_mask=None):
        embedded = self.embedding(input_ids)
        embedded = self.embedding_dropout(embedded)

        conv_input = embedded.transpose(1, 2)
        conv_outputs = [F.relu(conv(conv_input)) for conv in self.convs]

        min_len = min(x.size(2) for x in conv_outputs)
        conv_outputs = [x[:, :, :min_len] for x in conv_outputs]
        cnn_features = torch.cat(conv_outputs, dim=1).transpose(1, 2)

        if attention_mask is not None:
            attention_mask = attention_mask[:, :min_len]
            key_padding_mask = ~attention_mask.bool()
        else:
            key_padding_mask = None

        lstm_output, _ = self.bilstm(cnn_features)
        attn_output, _ = self.multihead_attn(
            lstm_output,
            lstm_output,
            lstm_output,
            key_padding_mask=key_padding_mask
        )

        attn_output = self.layer_norm(lstm_output + attn_output)
        pooled = self.attention_pooling(attn_output, attention_mask)
        return self.classifier(self.dropout(pooled))

In [17]:
#@title 2.6 Train Classifier

#----------------- calculate and show evaluation scores-------------
def evaluate_classifier(model, dataloader, device="cpu"):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device).long()

            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    weighted_f1 = f1_score(all_labels, all_preds, average="weighted", zero_division=0)

    print("Dev Accuracy:", round(acc, 4))
    print("Dev Macro F1:", round(macro_f1, 4))
    print("Dev Weighted F1:", round(weighted_f1, 4))
    print("\nClassification report:")
    print(classification_report(
        all_labels,
        all_preds,
        labels=list(range(4)),
        target_names=[ID2LABEL[i] for i in range(4)],
        zero_division=0
    ))
    print("Confusion matrix:")
    print(confusion_matrix(all_labels, all_preds, labels=list(range(4))))

    return acc, macro_f1, weighted_f1


def get_class_weights_from_dataset(dataset, num_labels, device):
    labels = []
    for item in dataset:
        labels.append(int(item["label"]))

    labels = np.array(labels)
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.arange(num_labels),
        y=labels
    )
    return torch.tensor(class_weights, dtype=torch.float).to(device)


def train_architecture(
    architecture_name: str,
    model_factory: Callable[[], nn.Module],
    train_claims,
    dev_claims,
    evidence_corpus,
    vocab,
    train_retriever,
    dev_retriever,
    train_evidence_mode="gold_and_retrieved",
    dev_evidence_mode="retrieved",
    use_class_weights=False,
    retrieval_top_k=4,
    epochs=8,
    batch_size=32,
    lr=5e-4,
    max_len=384,
    max_evidence=4,
    device="cpu"
):
    set_seed(42)
    print(f"\n===== Training {architecture_name} =====")

    train_dataset = ClaimEvidenceDataset(
        claims_json=train_claims,
        evidence_corpus=evidence_corpus,
        vocab=vocab,
        max_len=max_len,
        max_evidence=max_evidence,
        evidence_mode=train_evidence_mode,
        retriever=train_retriever,
        retrieval_top_k=retrieval_top_k,
        is_test=False
    )

    dev_dataset = ClaimEvidenceDataset(
        claims_json=dev_claims,
        evidence_corpus=evidence_corpus,
        vocab=vocab,
        max_len=max_len,
        max_evidence=max_evidence,
        evidence_mode=dev_evidence_mode,
        retriever=dev_retriever,
        retrieval_top_k=retrieval_top_k,
        is_test=False
    )

    generator = torch.Generator()
    generator.manual_seed(42)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn,
        generator=generator
    )

    dev_loader = DataLoader(
        dev_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn
    )

    model = model_factory().to(device)
    optimiser = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    if use_class_weights:
        class_weights = get_class_weights_from_dataset(train_dataset, num_labels=4, device=device)
        print("Class weights:", class_weights)
        criterion = nn.CrossEntropyLoss(weight=class_weights)
    else:
        criterion = nn.CrossEntropyLoss()

    best_macro_f1 = -1.0
    best_state_dict = None

    for epoch in range(epochs):
        print(f"\nEpoch {epoch + 1}/{epochs}")
        model.train()
        total_loss = 0.0

        for batch in tqdm(train_loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device).long()

            optimiser.zero_grad()
            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimiser.step()

            total_loss += loss.item()

        print("Training loss:", round(total_loss / max(len(train_loader), 1), 4))
        dev_acc, dev_macro_f1, dev_weighted_f1 = evaluate_classifier(model, dev_loader, device=device)
        print("Train evidence mode:", train_evidence_mode)
        print("Dev evidence mode:", dev_evidence_mode)

        if dev_macro_f1 > best_macro_f1:
            best_macro_f1 = dev_macro_f1
            best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            print("New best macro F1:", round(best_macro_f1, 4))

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    return model, best_macro_f1


def predict_claims(
    claims_json,
    evidence_corpus,
    retriever,
    model,
    vocab,
    output_path,
    retrieval_top_k=4,
    max_evidence=4,
    batch_size=32,
    max_len=384,
    is_test=False,
    device="cpu"
):
    dataset = ClaimEvidenceDataset(
        claims_json=claims_json,
        evidence_corpus=evidence_corpus,
        vocab=vocab,
        max_len=max_len,
        max_evidence=max_evidence,
        evidence_mode="retrieved",
        retriever=retriever,
        retrieval_top_k=retrieval_top_k,
        is_test=is_test
    )

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    model.eval()
    predictions = {}

    with torch.no_grad():
        for batch in tqdm(loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            pred_ids = torch.argmax(logits, dim=1).cpu().numpy().tolist()

            for claim_id, pred_id, evidence_ids in zip(batch["claim_ids"], pred_ids, batch["evidence_ids"]):
                predictions[claim_id] = {
                    "claim_text": claims_json[claim_id]["claim_text"],
                    "claim_label": ID2LABEL[pred_id],
                    "evidences": evidence_ids[:max_evidence]
                }

    save_json(predictions, output_path)
    print("Saved predictions to:", output_path)
    return predictions


In [69]:
COMMON_TRAIN_ARGS = dict(
    train_claims=train_claims,
    dev_claims=dev_claims,
    evidence_corpus=evidence_corpus,
    vocab=vocab,
    train_retriever=train_json_retriever,
    dev_retriever=dev_json_retriever,
    # Train with both gold evidence and retrieved evidence as input.
    # Keep dev realistic: use retrieved evidence only, because test has no gold evidence.
    train_evidence_mode="gold_and_retrieved",
    dev_evidence_mode="retrieved",
    retrieval_top_k=4,
    epochs=10,
    batch_size=64,
    lr=5e-4,
    max_len=384,
    max_evidence=4,
    device=device
)


def make_cnn_bilstm_multihead():
    return CNNBiLSTMMultiheadClassifier(
        vocab_size=len(vocab),
        embedding_dim=128,
        cnn_channels=64,
        kernel_sizes=(3, 5, 7),
        lstm_hidden_dim=128,
        lstm_layers=1,
        num_labels=4,
        num_heads=4,
        dropout=0.15,
        pad_idx=vocab["<PAD>"]
    )


cnn_bilstm_multihead, best_f1_2 = train_architecture(
    architecture_name="CNN + BiLSTM + multi-head attention",
    model_factory=make_cnn_bilstm_multihead,
    use_class_weights=True, # balanced class
    **COMMON_TRAIN_ARGS
)



print("\nBest dev macro F1 scores:")
print("CNN+BiLSTM+multihead:", round(best_f1_2, 4))



===== Training CNN + BiLSTM + multi-head attention =====
Class weights: tensor([0.5915, 1.5427, 0.7953, 2.4758], device='cuda:0')

Epoch 1/10


  0%|          | 0/20 [00:00<?, ?it/s]

Training loss: 1.4116


  0%|          | 0/3 [00:00<?, ?it/s]

Dev Accuracy: 0.2078
Dev Macro F1: 0.1446
Dev Weighted F1: 0.1278

Classification report:
                 precision    recall  f1-score   support

       SUPPORTS       0.00      0.00      0.00        68
        REFUTES       0.17      0.89      0.29        27
NOT_ENOUGH_INFO       0.57      0.20      0.29        41
       DISPUTED       0.00      0.00      0.00        18

       accuracy                           0.21       154
      macro avg       0.19      0.27      0.14       154
   weighted avg       0.18      0.21      0.13       154

Confusion matrix:
[[ 0 65  3  0]
 [ 0 24  3  0]
 [ 0 33  8  0]
 [ 0 18  0  0]]
Train evidence mode: gold_and_retrieved
Dev evidence mode: retrieved
New best macro F1: 0.1446

Epoch 2/10


...

Dev Accuracy: 0.487
Dev Macro F1: 0.4483
Dev Weighted F1: 0.4832

Classification report:
                 precision    recall  f1-score   support

       SUPPORTS       0.76      0.41      0.53        68
        REFUTES       0.41      0.33      0.37        27
NOT_ENOUGH_INFO       0.40      0.78      0.53        41
       DISPUTED       0.40      0.33      0.36        18

       accuracy                           0.49       154
      macro avg       0.49      0.46      0.45       154
   weighted avg       0.56      0.49      0.48       154

Confusion matrix:
[[28  7 31  2]
 [ 1  9 14  3]
 [ 4  1 32  4]
 [ 4  5  3  6]]
Train evidence mode: gold_and_retrieved
Dev evidence mode: retrieved
New best macro F1: 0.4483

Epoch 10/10


  0%|          | 0/20 [00:00<?, ?it/s]

Training loss: 0.7253


  0%|          | 0/3 [00:00<?, ?it/s]

Dev Accuracy: 0.5065
Dev Macro F1: 0.4479
Dev Weighted F1: 0.5079

Classification report:
                 precision    recall  f1-score   support

       SUPPORTS       0.72      0.53      0.61        68
        REFUTES       0.41      0.33      0.37        27
NOT_ENOUGH_INFO       0.43      0.68      0.53        41
       DISPUTED       0.29      0.28      0.29        18

       accuracy                           0.51       154
      macro avg       0.46      0.46      0.45       154
   weighted avg       0.54      0.51      0.51       154

Confusion matrix:
[[36  5 23  4]
 [ 3  9 12  3]
 [ 5  3 28  5]
 [ 6  5  2  5]]
Train evidence mode: gold_and_retrieved
Dev evidence mode: retrieved

Best dev macro F1 scores:
CNN+BiLSTM+multihead: 0.4483


# 3.Testing and Evaluation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [24]:
# @title 3.1 · Evaluate retriever-only on dev — Evidence F, Claim Accuracy, Harmonic Mean
import subprocess
import sys

SPLIT = "dev"
TOP_K = 4
BM25_TOP_K = 200

PRED_FILE = f"outputs/{SPLIT}-retriever-only-k{TOP_K}-bm25{BM25_TOP_K}.json"
GT_FILE = f"data/{SPLIT}-claims.json"

subprocess.check_call(
    [
        sys.executable,
        f"{PROJECT_ROOT}/eval.py",
        "--predictions", PRED_FILE,
        "--groundtruth", GT_FILE,
        "--verbose",
    ]
)


0

In [70]:
#@title 3.2 Evaluate with the whole pipeling (retriever+classifier) with eval.py
DEV_PRED_PATH = OUTPUT_DIR / "dev_predictions_cnn_bilstm_multihead_final.json"


dev_predictions = predict_claims(
    claims_json=dev_claims,
    evidence_corpus=evidence_corpus,
    retriever=dev_json_retriever,
    model=cnn_bilstm_multihead,
    vocab=vocab,
    output_path=DEV_PRED_PATH,
    retrieval_top_k=4,
    max_evidence=4,
    batch_size=32,
    max_len=384,
    is_test=False,
    device=device
)



  0%|          | 0/5 [00:00<?, ?it/s]

Saved predictions to: /content/COMP90042_2026/outputs/dev_predictions_cnn_bilstm_multihead_final.json


In [71]:
print("\n multihead self-attetion")
!python eval.py --predictions outputs/dev_predictions_cnn_bilstm_multihead_final.json --groundtruth data/dev-claims.json


 multihead self-attetion
Evidence Retrieval F-score (F)    = 0.2016646052360338
Claim Classification Accuracy (A) = 0.487012987012987
Harmonic Mean of F and A          = 0.28522281798093546


In [62]:
#@title 3.3 Output testing prediction json file

test_claims = load_json(TEST_PATH)
test_json_retriever = FixedJSONRetriever(TEST_RETRIEVER_JSON_PATH)

TEST_PRED_PATH_MODEL = OUTPUT_DIR / "test_predictions_cnn_bilstm_multihead_final.json"
test_predictions_model_1 = predict_claims(
      claims_json=test_claims,
      evidence_corpus=evidence_corpus,
      retriever=test_json_retriever,
      model=cnn_bilstm_multihead,
      vocab=vocab,
      output_path=TEST_PRED_PATH_MODEL,
      retrieval_top_k=4,
      max_evidence=4,
      batch_size=32,
      max_len=384,
      is_test=True,
      device=device
)

Loaded JSON retriever: /content/COMP90042_2026/outputs/test-retriever-only-k4-bm25200.json
Claims with retrieved evidence: 153


  0%|          | 0/5 [00:00<?, ?it/s]

Saved predictions to: /content/COMP90042_2026/outputs/test_predictions_cnn_bilstm_multihead_final.json


## Object Oriented Programming codes here

The OOP implementation lives in `src/` (importable modules) and `scripts/` (entry points):

| Module | Key class / function |
|--------|----------------------|
| `src/config.py` | `Config` dataclass — all hyperparameters and paths |
| `src/data_loader.py` | `Claim` dataclass, `load_claims`, `load_evidence` |
| `src/retriever_bm25.py` | `BM25Retriever` — first-stage lexical retrieval |
| `src/retriever_cross_enc.py` | `CrossEncoderHead(nn.Module)` — BERT reranker |
| `src/hard_negatives.py` | `build_training_pairs` — hard-negative mining |
| `src/evaluator.py` | `evaluate` — evidence F, claim accuracy, harmonic mean |
    